# Data Loading and Fixed Split

This notebook is the canonical place to inspect the extracted dataset and create the fixed train / validation / test split used by every evaluation notebook.


## Run Safety

Before running this notebook, restart the kernel.

Why this matters:
- previous model runs can leave tensors and cached arrays in memory
- old variables can leak into the current run and make comparisons unfair
- we want every notebook to save a clean, reproducible result

Recommended workflow:
1. Restart kernel
2. Run all cells from top to bottom
3. Let the notebook save its outputs before closing it


In [1]:
# Memory cleanup before starting this notebook.
import gc

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

print('Kernel memory cleanup complete. Start the notebook from here after a restart.')


Kernel memory cleanup complete. Start the notebook from here after a restart.


In [2]:
# Standard-library imports used throughout the evaluation notebooks.
import json
import math
import random
import sys
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from datetime import datetime

# Core scientific / ML stack.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

# ROC / PR curves are useful for classification comparison if sklearn is available.
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    from sklearn.preprocessing import label_binarize
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Resolve the project root robustly no matter where Jupyter was launched from.
SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = SEARCH_ROOT
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / '04_model_evaluation').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_ROOT = PROJECT_ROOT / '04_model_evaluation'
THESIS_ROOT = EVAL_ROOT
NOTEBOOK_ROOT = EVAL_ROOT / 'notebooks'
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'
PLOTS_ROOT = EVAL_ROOT / 'plots'
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))
ORIGINAL_DATASET_ROOT = PROJECT_ROOT / '03_dataset' / 'hybrid_maneuver_dataset'
PREFERRED_DATASET_ROOT = Path('/media/basudeo/1044063744061FD8/hybrid_maneuver_dataset')
DATASET_ROOT = PREFERRED_DATASET_ROOT if PREFERRED_DATASET_ROOT.exists() else ORIGINAL_DATASET_ROOT

print('Project root:', PROJECT_ROOT)
print('Evaluation workspace:', THESIS_ROOT)
print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred external dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Torch version:', torch.__version__)
print('Sklearn available for ROC/PR plots:', SKLEARN_AVAILABLE)


Project root: /home/basudeo/Documents/Thesis
Thesis workspace: /home/basudeo/Documents/Thesis/Thesis_Repo
Original dataset root: /home/basudeo/Documents/Thesis/hybrid_maneuver_dataset
Preferred external dataset root: /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset
Dataset root: /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset
Torch version: 2.3.1+cu118
Sklearn available for ROC/PR plots: True


In [3]:
# Shared experiment configuration.
# Keep this block explicit and heavily commented because it is the main place
# you will tune once more data has been collected.
LABEL_MODE = 'reduced'      # Use the balanced 5-class setting by default.
SEED = 42                   # Fix the split and initialization for reproducibility.
PAST_LEN = 10               # Number of past timesteps used to predict the current maneuver.
FUTURE_LEN = 5             # Number of future steps used for trajectory targets (ADE/FDE/RMSE).
SCAN_BEAMS = 512            # Number of lidar beams after resampling.
RANGE_CLIP = 30.0           # Maximum lidar range used for normalization.
BATCH_SIZE = 32             # Batch size for trainable models.
EPOCHS = 30                 # Upper bound; early stopping will usually stop sooner.
EARLY_STOPPING_PATIENCE = 5 # Patience on validation macro-F1.
LR = 1e-3                   # Adam learning rate.
WEIGHT_DECAY = 1e-4         # Small regularization on trainable models.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None          # Set to an integer for quick smoke tests.
USE_CPU = False             # Force CPU if needed.

# Hidden sizes for the current classification baselines.
CNN_HIDDEN = 96
GRAPH_HIDDEN = 96
FUSION_HIDDEN = 128
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MSG_PASSES = 2
DROPOUT = 0.10
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

# Device selection is kept simple and transparent.
device = torch.device('cuda' if torch.cuda.is_available() and not USE_CPU else 'cpu')
print('Device:', device)


Device: cuda


In [4]:
# Shared data and evaluation utilities live in one helper module so that
# all notebooks reuse the same dataset, split, path-remapping, and metric logic.
from dataset_helper import (
    SKLEARN_AVAILABLE,
    build_class_weights,
    build_label_mapping,
    build_run_manifest,
    build_sample_table,
    canonical_agent_order,
    compute_classification_metrics_from_probs,
    compute_trajectory_metrics,
    configure_helper,
    discover_frame_files,
    edge_features_for_order,
    group_streams,
    load_npy_cached,
    node_feature,
    prepare_result_dirs,
    remap_dataset_path,
    resample_scan,
    save_confusion_matrix,
    save_history_plot,
    save_mean_step_error_plot,
    save_or_load_fixed_split,
    save_predictions_csv,
    save_roc_pr_curves,
    save_run_manifest,
    save_training_history,
    save_trajectory_overlay_plots,
    set_seed,
    timestamp_tag,
)

configure_helper(
    dataset_root=DATASET_ROOT,
    original_dataset_root=ORIGINAL_DATASET_ROOT,
    results_root=RESULTS_ROOT,
    weights_root=WEIGHTS_ROOT,
)


# Inspect the Dataset

This section is deliberately descriptive. Before training anything, we verify the labels, the extracted runs, and how many balanced samples we actually have.


In [5]:
labels, label_mapping = build_label_mapping(LABEL_MODE)
print('Active labels:', labels)
print('Label mapping:', label_mapping)

frame_files = discover_frame_files(DATASET_ROOT)
assert frame_files, f'No frames.jsonl files found under {DATASET_ROOT}'
print('Found extracted datasets:')
for path in frame_files:
    print(' -', path)


Active labels: ['go_to_goal', 'avoid_left', 'avoid_right', 'commit_forward', 'arrived']
Label mapping: {'bootstrap': None, 'go_to_goal': 'go_to_goal', 'avoid_left': 'avoid_left', 'avoid_right': 'avoid_right', 'commit_forward': 'commit_forward', 'reverse': None, 'recover': None, 'reassess': None, 'arrived': 'arrived', 'stop': None}
Found extracted datasets:
 - /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset/run_model_20260411_160709/frames.jsonl
 - /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset/run_model_20260411_170338/frames.jsonl
 - /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset/run_model_20260411_173049/frames.jsonl
 - /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset/run_model_20260411_174655/frames.jsonl
 - /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset/run_model_20260415_194021/frames.jsonl
 - /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset/run_model_20260417_224523/frames.jsonl
 - /media/basudeo/1044063744061FD8/hybrid_maneuver_da

In [6]:
streams = group_streams(DATASET_ROOT, allowed_labels=set(labels), label_mapping=label_mapping)
print('Number of ego streams:', len(streams))

raw_label_counts = Counter()
mapped_label_counts = Counter()
for frames_path in frame_files:
    with frames_path.open() as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            raw_label = str(row['teacher']['label'])
            raw_label_counts[raw_label] += 1
            mapped = label_mapping.get(raw_label, raw_label)
            if mapped is not None:
                mapped_label_counts[mapped] += 1

print('Raw label counts:')
display(pd.DataFrame({'raw_label': list(raw_label_counts.keys()), 'count': list(raw_label_counts.values())}).sort_values('count', ascending=False))
print('Mapped training label counts:')
display(pd.DataFrame({'mapped_label': labels, 'count': [mapped_label_counts.get(label, 0) for label in labels]}).sort_values('count', ascending=False))


Number of ego streams: 14
Raw label counts:


,raw_label,count
0,go_to_goal,6170
3,commit_forward,2600
5,arrived,2017
1,avoid_left,475
4,avoid_right,422
2,reassess,336
8,recover,150
7,reverse,139
6,bootstrap,1


Mapped training label counts:


,mapped_label,count
0,go_to_goal,6170
3,commit_forward,2600
4,arrived,2017
1,avoid_left,475
2,avoid_right,422


In [7]:
# Peek at one raw frame and one raw scan so the data layout stays intuitive.
raw_frame = None
for frames_path in frame_files:
    with frames_path.open() as f:
        for line in f:
            if line.strip():
                raw_frame = json.loads(line)
                break
    if raw_frame is not None:
        break

print('Example frame keys:', sorted(raw_frame.keys()))
print('Example teacher block:')
print(json.dumps(raw_frame['teacher'], indent=2))
print('Example modalities block:')
print(json.dumps(raw_frame['modalities'], indent=2))

scan_ref = raw_frame['modalities']['ego_planar_scan']
scan_arr = load_npy_cached(str(scan_ref['path']))
print('Raw planar scan shape:', scan_arr.shape)
print('First five rows of the raw scan array:')
print(scan_arr[:5])
print('Processed scan shape:', resample_scan(scan_arr, SCAN_BEAMS, RANGE_CLIP).shape)


Example frame keys: ['agents', 'edges', 'ego_id', 'episode_id', 'modalities', 'readiness', 'teacher', 'timestamp_ns', 'world_name']
Example teacher block:
{
  "label": "go_to_goal",
  "label_source": "controller_state",
  "controller_state": "go_to_goal",
  "command": {
    "linear_x": 0.4,
    "angular_z": -0.85
  },
  "obstacle_action": "clear",
  "obstacle_clearance": null
}
Example modalities block:
{
  "ego_planar_scan": {
    "path": "/home/basudeo/Documents/Thesis/hybrid_maneuver_dataset/run_model_20260411_160709/assets/husky_local/planar_scan/1775916446703087395.npy",
    "timestamp_ns": 1775916446703087395,
    "modality": "planar_scan",
    "shape": [
      920,
      2
    ],
    "dtype": "float32"
  },
  "ego_front_pointcloud": null,
  "ego_front_camera": {
    "path": "/home/basudeo/Documents/Thesis/hybrid_maneuver_dataset/run_model_20260411_160709/assets/husky_local/front_image/1775916446634443181.npy",
    "timestamp_ns": 1775916446634443181,
    "modality": "image",
   

# Build and Save the Fixed Split

Every model notebook should evaluate on the exact same train / validation / test partition. This notebook creates the canonical split file used by the rest of the evaluation suite.


In [8]:
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)
split_path = SPLITS_ROOT / f'classification_split_{LABEL_MODE}_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json'
split_info = save_or_load_fixed_split(
    sample_table=sample_table,
    split_path=split_path,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
)

print('Split file:', split_path)
print('Total windowed samples:', split_info['sample_count'])
print('Future horizon:', split_info['future_len'])
print('Train samples:', len(split_info['train_indices']))
print('Validation samples:', len(split_info['val_indices']))
print('Test samples:', len(split_info['test_indices']))


Split file: /home/basudeo/Documents/Thesis/Thesis_Repo/splits/classification_split_reduced_seed42_past10_future5.json
Total windowed samples: 4063
Future horizon: 5
Train samples: 2844
Validation samples: 609
Test samples: 610


In [9]:
# Show a quick table of sample ids from each split for sanity.
split_preview = pd.DataFrame({
    'train_sample_id': [split_info['sample_ids'][idx] for idx in split_info['train_indices'][:5]],
    'val_sample_id': [split_info['sample_ids'][idx] for idx in split_info['val_indices'][:5]],
    'test_sample_id': [split_info['sample_ids'][idx] for idx in split_info['test_indices'][:5]],
})
display(split_preview)


,train_sample_id,val_sample_id,test_sample_id
0,run_model_20260417_224523::husky_2::1776458761...,run_model_20260417_225909::husky_local::177646...,run_model_20260417_225909::husky_2::1776460230...
1,run_model_20260415_194021::husky_local::177627...,run_model_20260415_194021::husky_local::177627...,run_model_20260417_225909::husky_2::1776460305...
2,run_model_20260415_194021::husky_2::1776274953...,run_model_20260417_225909::husky_2::1776459799...,run_model_20260415_194021::husky_local::177627...
3,run_model_20260417_225909::husky_2::1776460081...,run_model_20260415_194021::husky_2::1776274910...,run_model_20260417_224523::husky_2::1776458881...
4,run_model_20260417_225909::husky_local::177646...,run_model_20260417_224523::husky_local::177645...,run_model_20260417_224523::husky_2::1776459238...


In [10]:
# Inspect one trajectory target so future-metric supervision stays easy to understand.
preview_meta = sample_table[0]
print('Preview sample id:', preview_meta['sample_id'])
print('Preview label:', preview_meta['label'])
print('Future relative XY target shape:', np.asarray(preview_meta['future_xy']).shape)
display(pd.DataFrame({
    'dt_seconds': preview_meta['future_dt'],
    'future_dx': [xy[0] for xy in preview_meta['future_xy']],
    'future_dy': [xy[1] for xy in preview_meta['future_xy']],
}))


Preview sample id: stream000_start00000
Preview label: avoid_left
Future relative XY target shape: (5, 2)


,dt_seconds,future_dx,future_dy
0,0.485593,-0.001448,0.017917
1,0.933046,-0.010338,0.051020
2,1.423051,-0.040485,0.122844
3,1.857704,-0.076761,0.216027
4,2.368364,-0.103421,0.291451
